In [26]:
import os
import random

import numpy as np
import mne
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

In [27]:
DATA_FOLDER = "../data/BCI_IV_2a"

SUBJECTS = [
    "A01",
    "A02",
    "A03",
    "A04",
    "A05",
    "A06",
    "A07",
    "A08",
    "A09"
]

SFREQ = 250

LOW_FREQ = 8
HIGH_FREQ = 30

TMIN = 1
TMAX = 3

N_CHANNELS = 22
N_CLASSES = 4

BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

RANDOM_STATE = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cpu


In [28]:
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

In [29]:
def load_training_subject(subject):

    file_path = os.path.join(
        DATA_FOLDER,
        subject + "T.gdf"
    )

    print("\nLoading:", file_path)

    raw = mne.io.read_raw_gdf(
        file_path,
        preload=True,
        verbose=False
    )

    # Keep EEG channels only
    raw.pick("eeg")

    print("EEG channels:", len(raw.ch_names))
    print("Sampling frequency:", raw.info["sfreq"])

    # Band-pass filter
    raw.filter(
        l_freq=LOW_FREQ,
        h_freq=HIGH_FREQ,
        verbose=False
    )

    # Get annotations/events
    events, event_dict = mne.events_from_annotations(
        raw,
        verbose=False
    )

    print("Event dictionary:")
    print(event_dict)

    # Standard BCI IV 2a motor-imagery events
    required_events = [
        "769",  # Left hand
        "770",  # Right hand
        "771",  # Foot
        "772"   # Tongue
    ]

    missing = [
        event for event in required_events
        if event not in event_dict
    ]

    if missing:

        raise ValueError(
            f"{subject}: Missing motor-imagery events: {missing}\n"
            f"Available events: {list(event_dict.keys())}"
        )

    # MNE event IDs
    event_id = {
        "LEFT": event_dict["769"],
        "RIGHT": event_dict["770"],
        "FOOT": event_dict["771"],
        "TONGUE": event_dict["772"]
    }

    print("Using event IDs:")
    print(event_id)

    # Create epochs
    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        picks="eeg",
        preload=True,
        reject_by_annotation=False,
        verbose=False
    )

    X = epochs.get_data()

    # MNE returns the event IDs:
    # LEFT   -> 769
    # RIGHT  -> 770
    # FOOT   -> 771
    # TONGUE -> 772

    # Convert to our labels:
    # LEFT   -> 0
    # RIGHT  -> 1
    # FOOT   -> 2
    # TONGUE -> 3

    event_code_to_label = {
        event_dict["769"]: 0,
        event_dict["770"]: 1,
        event_dict["771"]: 2,
        event_dict["772"]: 3
    }

    y = np.array([
        event_code_to_label[event]
        for event in epochs.events[:, -1]
    ])

    # Keep the first 22 EEG channels
    X = X[:, :N_CHANNELS, :]

    print("X shape:", X.shape)
    print("y shape:", y.shape)

    print(
        "Class counts:",
        np.unique(y, return_counts=True)
    )

    return X, y

In [30]:
def normalize_epochs(X):

    mean = X.mean(
        axis=2,
        keepdims=True
    )

    std = X.std(
        axis=2,
        keepdims=True
    )

    X_norm = (
        X - mean
    ) / (
        std + 1e-6
    )

    return X_norm.astype(np.float32)

In [31]:
subject_data = {}

for subject in SUBJECTS:

    print("\n" + "=" * 50)
    print("SUBJECT:", subject)
    print("=" * 50)

    X, y = load_training_subject(subject)

    X = normalize_epochs(X)

    subject_data[subject] = {
        "X": X,
        "y": y
    }

    print("Final X:", X.shape)
    print("Final y:", y.shape)


SUBJECT: A01

Loading: ../data/BCI_IV_2a\A01T.gdf


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channels: 25
Sampling frequency: 250.0
Event dictionary:
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using event IDs:
{'LEFT': 7, 'RIGHT': 8, 'FOOT': 9, 'TONGUE': 10}
X shape: (288, 22, 501)
y shape: (288,)
Class counts: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Final X: (288, 22, 501)
Final y: (288,)

SUBJECT: A02

Loading: ../data/BCI_IV_2a\A02T.gdf


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channels: 25
Sampling frequency: 250.0
Event dictionary:
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using event IDs:
{'LEFT': 7, 'RIGHT': 8, 'FOOT': 9, 'TONGUE': 10}
X shape: (288, 22, 501)
y shape: (288,)
Class counts: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Final X: (288, 22, 501)
Final y: (288,)

SUBJECT: A03

Loading: ../data/BCI_IV_2a\A03T.gdf


...

EEG channels: 25
Sampling frequency: 250.0
Event dictionary:
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using event IDs:
{'LEFT': 7, 'RIGHT': 8, 'FOOT': 9, 'TONGUE': 10}
X shape: (288, 22, 501)
y shape: (288,)
Class counts: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Final X: (288, 22, 501)
Final y: (288,)

SUBJECT: A08

Loading: ../data/BCI_IV_2a\A08T.gdf


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channels: 25
Sampling frequency: 250.0
Event dictionary:
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using event IDs:
{'LEFT': 7, 'RIGHT': 8, 'FOOT': 9, 'TONGUE': 10}
X shape: (288, 22, 501)
y shape: (288,)
Class counts: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Final X: (288, 22, 501)
Final y: (288,)

SUBJECT: A09

Loading: ../data/BCI_IV_2a\A09T.gdf


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


EEG channels: 25
Sampling frequency: 250.0
Event dictionary:
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Using event IDs:
{'LEFT': 7, 'RIGHT': 8, 'FOOT': 9, 'TONGUE': 10}
X shape: (288, 22, 501)
y shape: (288,)
Class counts: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Final X: (288, 22, 501)
Final y: (288,)


In [33]:
class EEGNet(nn.Module):

    def __init__(
        self,
        n_channels=22,
        n_samples=501,
        n_classes=4,
        dropout=0.5
    ):

        super().__init__()

        # --------------------------------
        # Temporal convolution
        # --------------------------------

        self.temporal = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=(1, 64),
                padding=(0, 32),
                bias=False
            ),

            nn.BatchNorm2d(16)
        )

        # --------------------------------
        # Depthwise spatial convolution
        # --------------------------------

        self.spatial = nn.Sequential(

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=(n_channels, 1),
                groups=16,
                bias=False
            ),

            nn.BatchNorm2d(32),

            nn.ELU(),

            nn.AvgPool2d(
                kernel_size=(1, 4)
            ),

            nn.Dropout(dropout)
        )

        # --------------------------------
        # Separable convolution
        # --------------------------------

        self.separable = nn.Sequential(

            nn.Conv2d(
                in_channels=32,
                out_channels=32,
                kernel_size=(1, 16),
                padding=(0, 8),
                groups=32,
                bias=False
            ),

            nn.Conv2d(
                in_channels=32,
                out_channels=32,
                kernel_size=(1, 1),
                bias=False
            ),

            nn.BatchNorm2d(32),

            nn.ELU(),

            nn.AvgPool2d(
                kernel_size=(1, 8)
            ),

            nn.Dropout(dropout)
        )

        # --------------------------------
        # Automatically determine size
        # --------------------------------

        with torch.no_grad():

            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_samples
            )

            x = self.temporal(dummy)
            x = self.spatial(x)
            x = self.separable(x)

            flattened_size = x.flatten(
                start_dim=1
            ).shape[1]

        self.classifier = nn.Linear(
            flattened_size,
            n_classes
        )

    def forward(self, x):

        x = self.temporal(x)

        x = self.spatial(x)

        x = self.separable(x)

        x = x.flatten(
            start_dim=1
        )

        x = self.classifier(x)

        return x

In [34]:
model = EEGNet(
    n_channels=N_CHANNELS,
    n_samples=501,
    n_classes=N_CLASSES
)

print(model)

dummy_input = torch.randn(
    4,
    1,
    N_CHANNELS,
    501
)

output = model(dummy_input)

print("\nInput shape :", dummy_input.shape)
print("Output shape:", output.shape)

EEGNet(
  (temporal): Sequential(
    (0): Conv2d(1, 16, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (spatial): Sequential(
    (0): Conv2d(16, 32, kernel_size=(22, 1), stride=(1, 1), groups=16, bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (4): Dropout(p=0.5, inplace=False)
  )
  (separable): Sequential(
    (0): Conv2d(32, 32, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=32, bias=False)
    (1): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (cl

In [35]:
def train_eegnet(
    X_train,
    y_train,
    epochs=EPOCHS
):

    # -----------------------------
    # Convert to PyTorch tensors
    # -----------------------------

    X_tensor = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_train,
        dtype=torch.long
    )

    # EEGNet input:
    # batch x 1 x channels x samples

    X_tensor = X_tensor.unsqueeze(1)

    # -----------------------------
    # Dataset
    # -----------------------------

    dataset = TensorDataset(
        X_tensor,
        y_tensor
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    # -----------------------------
    # Model
    # -----------------------------

    model = EEGNet(
        n_channels=N_CHANNELS,
        n_samples=X_train.shape[2],
        n_classes=N_CLASSES
    ).to(DEVICE)

    # -----------------------------
    # Loss
    # -----------------------------

    criterion = nn.CrossEntropyLoss()

    # -----------------------------
    # Optimizer
    # -----------------------------

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # -----------------------------
    # Training
    # -----------------------------

    loss_history = []

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(X_batch)

            loss = criterion(
                outputs,
                y_batch
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = (
            running_loss / len(loader)
        )

        loss_history.append(epoch_loss)

        if (
            (epoch + 1) % 10 == 0
            or epoch == 0
        ):

            print(
                f"Epoch {epoch + 1:03d}/{epochs} "
                f"| Loss: {epoch_loss:.4f}"
            )

    return model, loss_history

In [37]:
def predict_eegnet(model, X):

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )

    X_tensor = X_tensor.unsqueeze(1)

    dataset = TensorDataset(
        X_tensor
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model.eval()

    predictions = []

    with torch.no_grad():

        for (X_batch,) in loader:

            X_batch = X_batch.to(DEVICE)

            outputs = model(X_batch)

            preds = torch.argmax(
                outputs,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

    return np.array(predictions)

In [38]:
fold_results = []

all_true = []
all_pred = []

for test_subject in SUBJECTS:

    print("\n")
    print("=" * 60)
    print("HELD-OUT SUBJECT:", test_subject)
    print("=" * 60)

    # --------------------------------
    # Training = other 8 subjects
    # --------------------------------

    X_train_list = []
    y_train_list = []

    for subject in SUBJECTS:

        if subject == test_subject:
            continue

        X_train_list.append(
            subject_data[subject]["X"]
        )

        y_train_list.append(
            subject_data[subject]["y"]
        )

    X_train = np.concatenate(
        X_train_list,
        axis=0
    )

    y_train = np.concatenate(
        y_train_list,
        axis=0
    )

    # --------------------------------
    # Testing = held-out subject
    # --------------------------------

    X_test = subject_data[
        test_subject
    ]["X"]

    y_test = subject_data[
        test_subject
    ]["y"]

    print(
        "Training:",
        X_train.shape
    )

    print(
        "Testing :",
        X_test.shape
    )

    print(
        "Training labels:",
        np.unique(y_train)
    )

    print(
        "Testing labels:",
        np.unique(y_test)
    )

    # --------------------------------
    # Train
    # --------------------------------

    model, loss_history = train_eegnet(
        X_train,
        y_train
    )

    # --------------------------------
    # Predict
    # --------------------------------

    y_pred = predict_eegnet(
        model,
        X_test
    )

    # --------------------------------
    # Metrics
    # --------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    fold_results.append({
        "subject": test_subject,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

    all_true.extend(
        y_test
    )

    all_pred.extend(
        y_pred
    )

    print(
        f"\n{test_subject} Accuracy: "
        f"{accuracy * 100:.2f}%"
    )

    print(
        f"{test_subject} Macro F1: "
        f"{macro_f1:.4f}"
    )



HELD-OUT SUBJECT: A01
Training: (2304, 22, 501)
Testing : (288, 22, 501)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]
Epoch 001/50 | Loss: 1.4016
Epoch 010/50 | Loss: 1.2668
Epoch 020/50 | Loss: 1.2032
Epoch 030/50 | Loss: 1.1505
Epoch 040/50 | Loss: 1.1321
Epoch 050/50 | Loss: 1.0973

A01 Accuracy: 53.47%
A01 Macro F1: 0.5241


HELD-OUT SUBJECT: A02
Training: (2304, 22, 501)
Testing : (288, 22, 501)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]
Epoch 001/50 | Loss: 1.3947
Epoch 010/50 | Loss: 1.2338
Epoch 020/50 | Loss: 1.1584
Epoch 030/50 | Loss: 1.1178
Epoch 040/50 | Loss: 1.0636
Epoch 050/50 | Loss: 1.0142

A02 Accuracy: 26.74%
A02 Macro F1: 0.1934


HELD-OUT SUBJECT: A03
Training: (2304, 22, 501)
Testing : (288, 22, 501)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]
Epoch 001/50 | Loss: 1.4046
Epoch 010/50 | Loss: 1.2811
Epoch 020/50 | Loss: 1.2265
Epoch 030/50 | Loss: 1.1651
Epoch 040/50 | Loss: 1.1459
Epoch 050/50 | Loss: 1.0942

A03 Accuracy: 51.74%
A03 

In [39]:
accuracies = np.array([
    result["accuracy"]
    for result in fold_results
])

f1_scores = np.array([
    result["macro_f1"]
    for result in fold_results
])

mean_accuracy = accuracies.mean()
std_accuracy = accuracies.std()

mean_f1 = f1_scores.mean()

print("\n")
print("=" * 50)
print("FINAL EEGNET LOSO RESULTS")
print("=" * 50)

for result in fold_results:

    print(
        f"{result['subject']}: "
        f"{result['accuracy'] * 100:.2f}% | "
        f"F1: {result['macro_f1']:.4f}"
    )

print("\n")

print(
    f"Mean Accuracy: "
    f"{mean_accuracy * 100:.2f}%"
)

print(
    f"Std Accuracy: "
    f"{std_accuracy * 100:.2f}%"
)

print(
    f"Mean Macro F1: "
    f"{mean_f1:.4f}"
)



FINAL EEGNET LOSO RESULTS
A01: 53.47% | F1: 0.5241
A02: 26.74% | F1: 0.1934
A03: 51.74% | F1: 0.4758
A04: 36.81% | F1: 0.3334
A05: 25.00% | F1: 0.1257
A06: 28.12% | F1: 0.2568
A07: 31.60% | F1: 0.2973
A08: 49.65% | F1: 0.4892
A09: 46.18% | F1: 0.4199


Mean Accuracy: 38.81%
Std Accuracy: 10.86%
Mean Macro F1: 0.3462


In [40]:
cm = confusion_matrix(
    all_true,
    all_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[347 158  72  71]
 [178 311  78  81]
 [205 174 128 141]
 [173 129 126 220]]


In [41]:
print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3],
        target_names=[
            "Left Hand",
            "Right Hand",
            "Foot",
            "Tongue"
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

   Left Hand       0.38      0.54      0.45       648
  Right Hand       0.40      0.48      0.44       648
        Foot       0.32      0.20      0.24       648
      Tongue       0.43      0.34      0.38       648

    accuracy                           0.39      2592
   macro avg       0.38      0.39      0.38      2592
weighted avg       0.38      0.39      0.38      2592

